# Part 6— Recommendation System


### Goal  
Recommend products to customers. Build a simple recommendation approach using 'customer purchase history'.

In [127]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [128]:
rules = rules = pd.read_pickle('strong_rules.pkl')  #to be readable as antecedents and consequents not as string
rules.head()

,antecedents,consequents,support,confidence,lift
364,frozenset({REGENCY TEA PLATE GREEN }),frozenset({REGENCY TEA PLATE PINK}),0.010898,0.748148,61.909259
365,frozenset({REGENCY TEA PLATE PINK}),frozenset({REGENCY TEA PLATE GREEN }),0.010898,0.901786,61.909259
588,frozenset({POPPY'S PLAYHOUSE LIVINGROOM }),"frozenset({POPPY'S PLAYHOUSE KITCHEN, POPPY'S ...",0.010035,0.738095,53.863517
583,"frozenset({POPPY'S PLAYHOUSE KITCHEN, POPPY'S ...",frozenset({POPPY'S PLAYHOUSE LIVINGROOM }),0.010035,0.732283,53.863517
358,frozenset({REGENCY SUGAR BOWL GREEN}),frozenset({REGENCY MILK JUG PINK }),0.011114,0.768657,52.381694


In [129]:
df = pd.read_csv('../data/online_retail_cleaned.csv')
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,transactiontype
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,Purchase
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,Purchase
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,Purchase
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,Purchase
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,Purchase
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,15.30,Purchase
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,25.50,Purchase
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,11.10,Purchase
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,11.10,Purchase
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom,54.08,Purchase


choose one customer, select columns : customerid, description, and print the set of previous product purchased by him

In [130]:
df['CustomerID'].dtypes

dtype('float64')

#### 1.Find the list of purchased products by one customer

In [131]:
customer_id = 13047.0  #choose a customer

customer_products = df.loc[df['CustomerID']== customer_id , 'Description'].dropna().unique()

print("Customer's previous purchases:")
print(customer_products.unique())

Customer's previous purchases:
<StringArray>
[     'ASSORTED COLOUR BIRD ORNAMENT',         'POPPY'S PLAYHOUSE BEDROOM ',
          'POPPY'S PLAYHOUSE KITCHEN',  'FELTCRAFT PRINCESS CHARLOTTE DOLL',
            'IVORY KNITTED MUG COSY ', 'BOX OF 6 ASSORTED COLOUR TEASPOONS',
      'BOX OF VINTAGE JIGSAW BLOCKS ',     'BOX OF VINTAGE ALPHABET BLOCKS',
           'HOME BUILDING BLOCK WORD',           'LOVE BUILDING BLOCK WORD',
 ...
     'DRAWER KNOB VINTAGE GLASS BALL',       'CARD HOLDER LOVE BIRD LARGE ',
        'FOOT STOOL HOME SWEET HOME ',                    'PETIT TRAY CHIC',
           'SWEET HEART CAKE CARRIER',         'PACK OF SIX LED TEA LIGHTS',
   '3 RAFFIA RIBBONS 50'S CHRISTMAS ',    'PAPER CHAIN KIT 50'S CHRISTMAS ',
          'SCANDINAVIAN REDS RIBBONS',                             'Manual']
Length: 106, dtype: str


The recommendation sysmtem aims to recommed not purchased product based on purchased product by one customer, so using customerid, product, and association rules we can find the list of purchased products (as antecents in the association) to recommend products (not purchased) from the consequents list 

#### 2. Find recommendations(Top 5)

In [132]:
rules.head()

,antecedents,consequents,support,confidence,lift
364,frozenset({REGENCY TEA PLATE GREEN }),frozenset({REGENCY TEA PLATE PINK}),0.010898,0.748148,61.909259
365,frozenset({REGENCY TEA PLATE PINK}),frozenset({REGENCY TEA PLATE GREEN }),0.010898,0.901786,61.909259
588,frozenset({POPPY'S PLAYHOUSE LIVINGROOM }),"frozenset({POPPY'S PLAYHOUSE KITCHEN, POPPY'S ...",0.010035,0.738095,53.863517
583,"frozenset({POPPY'S PLAYHOUSE KITCHEN, POPPY'S ...",frozenset({POPPY'S PLAYHOUSE LIVINGROOM }),0.010035,0.732283,53.863517
358,frozenset({REGENCY SUGAR BOWL GREEN}),frozenset({REGENCY MILK JUG PINK }),0.011114,0.768657,52.381694


In [133]:
antecedents = set(rules['antecedents'])


In [134]:
def recommend_products(customer_id, df, rules,  top_n=5):

    # list of purchased products
    purchased_products = set(
        df.loc[df['CustomerID'] == customer_id, 'Description'].dropna()
    )

    recommendations = []

    # use association rule, split antencents, consequents
    for _, rules in rules.iterrows():

        antecedents = set(rules['antecedents'])
        consequents = set(rules['consequents'])

        
        if antecedents.issubset(purchased_products):

            for product in consequents:

                # Do not recommend products already purchased
                if product not in purchased_products:

                    recommendations.append({
                        'Product': product,
                        'Confidence': rules['confidence'],
                        'Lift': rules['lift'],
                        'Support': rules['support'],
                        'Based_on': ', '.join(antecedents)
                    })

    # Convert recommendations to a DataFrame
    recommendations_df = pd.DataFrame(recommendations)

    if recommendations_df.empty:
        return recommendations_df

    # show the strongest rules for each recommended product
    recommendations_df = (
        recommendations_df
        .sort_values(
            ['Confidence', 'Lift', 'Support'],
            ascending=False
        )
        .drop_duplicates(subset='Product')
        .head(top_n)
        .reset_index(drop=True)
    )

    return recommendations_df

In [140]:
top5 =recommend_products(
    customer_id = 13047.0, df = df, rules= rules
)
print(f"Top 5 products recommended : \n, {top5}")


Top 5 products recommended : 
,                              Product  Confidence       Lift   Support  \
0           REGENCY SUGAR BOWL GREEN    0.757353  52.381694  0.011114   
1      POPPY'S PLAYHOUSE LIVINGROOM     0.732283  53.863517  0.010035   
2  SET OF 12 FAIRY CAKE BAKING CASES    0.694158  27.670783  0.010898   
3       FELTCRAFT PRINCESS LOLA DOLL    0.593516  29.104276  0.012840   
4   ROSES REGENCY TEACUP AND SAUCER     0.536723  12.705876  0.010250   

                                            Based_on  
0                             REGENCY MILK JUG PINK   
1  POPPY'S PLAYHOUSE KITCHEN, POPPY'S PLAYHOUSE B...  
2                   SET OF 6 SNACK LOAF BAKING CASES  
3                  FELTCRAFT PRINCESS CHARLOTTE DOLL  
4                              REGENCY TEAPOT ROSES   


In [139]:
print(type(rules['consequents'].iloc[0]), type(rules['antecedents'].iloc[0]))  #type of antecedents and consequents not str 

<class 'frozenset'> <class 'frozenset'>


## Recommendation Discussion


Cold Start Problem
What happens when:

A new customer has no history? How can we solve it?

New customer(frequency =1) means insufficient antecedents and purchased products to recommend associated products, so :  
- Popular products: recommend high frequent products purchased for costomers.  
- Content-based recommendation : recommend products based on features, description defined by the customer.  
- Hybrid systems : combine these 2 methods  to recommed products based on frequency and description.

### Evaluation
How would the company know recommendations are successful:
we can evalute models using some metrices focusing of customer's behavior on consequents recommended list :
- click rate : calculate the percentage of recommended products that customer clicks on it.
- Purchase rate : caculate the percentage of recommende products purchased by the customer.
- Customer engagement : measure the interaction of the  customer with recommended products, as viewing it. 
